In [1]:
from pyspark.sql import SparkSession

# Initialise Spark session
spark = (
    SparkSession.builder.appName("Rideshare_Analysis")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)

your 131072x1 screen size is bogus. expect trouble
24/08/23 21:25:22 WARN Utils: Your hostname, LAPTOP-354CCOC2 resolves to a loopback address: 127.0.1.1; using 172.21.14.166 instead (on interface eth0)
24/08/23 21:25:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/08/23 21:25:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# Load FHVHV data
fhvhv_df = spark.read.parquet("../data/raw")

# Load weather data
weather_df = spark.read.csv('../data/raw_csv/NYC hourly weather 2023-05-01 to 2024-05-01.csv', header=True, inferSchema=True)

# Load public holidays data
holidays_2023_df = spark.read.csv('../data/raw_csv/2023_holidays.csv', header=True, inferSchema=True)
holidays_2024_df = spark.read.csv('../data/raw_csv/2024_holidays.csv', header=True, inferSchema=True)

# Combine 2023 and 2024 holidays
holidays_df = holidays_2023_df.union(holidays_2024_df)


In [4]:
# Examine the initial shape of all the datasets

# Add the 'scripts' directory to the system path
import sys
sys.path.append("../scripts")

from functions import get_dataset_shape  

# Call functions to get an idea of dataset sizes
get_dataset_shape(fhvhv_df)
get_dataset_shape(weather_df)
get_dataset_shape(holidays_df)

Dataset shape is: 116706029 rows, 24 columns
Dataset shape is: 8808 rows, 24 columns
Dataset shape is: 26 rows, 2 columns


In [12]:
# Display schemas for all datasets
fhvhv_df.printSchema()
weather_df.printSchema()
holidays_df.printSchema()

# Preview a few records
fhvhv_df.show(7)
weather_df.show(7)
holidays_df.show(7)

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_

+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|   request_datetime|  on_scene_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|
+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+--

In [9]:
# Count missing values in each column and display as a table
from pyspark.sql.functions import col, sum

# Add the 'scripts' directory to the system path
import sys
sys.path.append("../scripts")

from functions import get_missing_value_counts  

# Call function to inspect missing values
get_missing_value_counts(fhvhv_df)
get_missing_value_counts(weather_df)
get_missing_value_counts(holidays_df)



+-----------------------+--------------------------+--------------------------+----------------------+-----------------------+---------------------+----------------------+------------------+------------------+----------------+---------------+-------------------------+-----------+---------+---------------+--------------------------+-----------------+----------+----------------+-------------------------+-----------------------+------------------------+----------------------+--------------------+
|hvfhs_license_num_count|dispatching_base_num_count|originating_base_num_count|request_datetime_count|on_scene_datetime_count|pickup_datetime_count|dropoff_datetime_count|PULocationID_count|DOLocationID_count|trip_miles_count|trip_time_count|base_passenger_fare_count|tolls_count|bcf_count|sales_tax_count|congestion_surcharge_count|airport_fee_count|tips_count|driver_pay_count|shared_request_flag_count|shared_match_flag_count|access_a_ride_flag_count|wav_request_flag_count|wav_match_flag_count|
+-

In [32]:
# Summary statistics
sample_df = fhvhv_df.sample(fraction=0.10, seed=42)  # 10% of the data
sample_df.describe().show()

sample_df.printSchema()


24/08/24 02:04:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-----------------+--------------------+--------------------+------------------+------------------+-----------------+-----------------+-------------------+------------------+------------------+-----------------+--------------------+-------------------+------------------+------------------+-------------------+-----------------+------------------+----------------+--------------+
|summary|hvfhs_license_num|dispatching_base_num|originating_base_num|      PULocationID|      DOLocationID|       trip_miles|        trip_time|base_passenger_fare|             tolls|               bcf|        sales_tax|congestion_surcharge|        airport_fee|              tips|        driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|
+-------+-----------------+--------------------+--------------------+------------------+------------------+-----------------+-----------------+-------------------+------------------+------------------+-----------------+-----

This next section will work on cleaning/preprocessing the 'FHVHV' dataset

In [34]:
# Based on contextual knowledge, 
# drop feature columns deemed unuseful or unaligned with the research purpose

columns_to_drop = ['dispatching_base_num', 'DropOff_datetime', 'DOLocationID', 'originating_base_num', 'request_datetime', 'on_scene_datetime', 'base_passenger_fare', 'tolls', 
                   'bcf', 'sales_tax', 'congestion_surcharge', 'airport_fee', 'shared_request_flag', 'shared_match_flag', 'access_a_ride_flag', 'wav_request_flag', 'wav_match_flag']

fhvhv_df = fhvhv_df.drop(*columns_to_drop)
get_dataset_shape(fhvhv_df)
fhvhv_df.limit(7)


Dataset shape is: 116700743 rows, 7 columns


hvfhs_license_num,pickup_datetime,PULocationID,trip_miles,trip_time,tips,driver_pay
HV0005,2023-05-01 00:09:39,32,2.247,685,0.0,10.27
HV0005,2023-05-01 00:43:35,81,1.826,493,0.0,7.75
HV0003,2023-05-01 00:06:54,121,3.88,810,1.0,12.71
HV0003,2023-05-01 00:24:27,92,2.31,694,0.0,10.57
HV0003,2023-05-01 00:39:13,53,9.45,1074,0.0,24.27
HV0003,2023-05-01 00:29:14,234,1.65,526,0.0,7.12
HV0003,2023-05-01 00:46:20,113,11.57,1886,0.0,32.93
HV0003,2023-05-01 00:00:59,243,0.9,323,0.0,5.38
HV0003,2023-05-01 00:26:18,166,1.39,452,0.0,6.08
HV0003,2023-05-01 00:59:55,164,2.77,587,1.0,9.16


In [20]:
from pyspark.sql import functions as F

# 'hvfhs_license_num' feature
# Filter out any invalid licenses that are not part of the big 4 rideshare companies
fhvhv_df = fhvhv_df.filter((F.col('hvfhs_license_num') == 'HV0003') | 
                (F.col('hvfhs_license_num') == 'HV0003') | 
                (F.col('hvfhs_license_num') == 'HV0003') | 
                (F.col('hvfhs_license_num') == 'HV0005'))

# Verify any changes to dataset size
get_dataset_shape(fhvhv_df)


Dataset shape is: 116706029 rows, 24 columns


In [41]:
from pyspark.ml.feature import OneHotEncoder, StringIndexer

# 'hvfhs_license_num' feature
# Convert Strings in columns to a numeric index
indexer = StringIndexer(inputCol="hvfhs_license_num", outputCol="license_index")
fhvhv_df = indexer.fit(fhvhv_df).transform(fhvhv_df)

# Convert numeric indices to a one hot encoded vector
encoder = OneHotEncoder(inputCol="license_index", outputCol="license_vec")
fhvhv_df = encoder.fit(fhvhv_df).transform(fhvhv_df)

fhvhv_df.limit(7)

hvfhs_license_num,pickup_datetime,PULocationID,trip_miles,trip_time,tips,driver_pay,earnings_per_hour,license_index,license_vec
HV0005,2023-05-01 00:09:39,32,2.247,685,0.0,10.27,53.97,1.0,"(1,[],[])"
HV0005,2023-05-01 00:43:35,81,1.826,493,0.0,7.75,56.59,1.0,"(1,[],[])"
HV0003,2023-05-01 00:06:54,121,3.88,810,1.0,12.71,60.93,0.0,"(1,[0],[1.0])"
HV0003,2023-05-01 00:24:27,92,2.31,694,0.0,10.57,54.83,0.0,"(1,[0],[1.0])"
HV0003,2023-05-01 00:39:13,53,9.45,1074,0.0,24.27,81.35,0.0,"(1,[0],[1.0])"
HV0003,2023-05-01 00:29:14,234,1.65,526,0.0,7.12,48.73,0.0,"(1,[0],[1.0])"
HV0003,2023-05-01 00:46:20,113,11.57,1886,0.0,32.93,62.86,0.0,"(1,[0],[1.0])"


In [ ]:
from pyspark.sql.functions import col

fhvhv_df = fhvhv_df.filter((col("pickup_datetime") >= "2023-05-01") & (col("pickup_datetime") < "2023-11-01"))


In [ ]:
from pyspark.sql.functions import dayofweek, hour

fhvhv_df = fhvhv_df.withColumn("day_of_week", dayofweek("pickup_datetime"))
fhvhv_df = fhvhv_df.withColumn("hour_of_day", hour("pickup_datetime"))

# One-hot encoding day of the week and hour
indexer_day = StringIndexer(inputCol="day_of_week", outputCol="day_index")
fhvhv_df = indexer_day.fit(fhvhv_df).transform(fhvhv_df)

encoder_day = OneHotEncoder(inputCol="day_index", outputCol="day_vec")
fhvhv_df = encoder_day.fit(fhvhv_df).transform(fhvhv_df)

indexer_hour = StringIndexer(inputCol="hour_of_day", outputCol="hour_index")
fhvhv_df = indexer_hour.fit(fhvhv_df).transform(fhvhv_df)

encoder_hour = OneHotEncoder(inputCol="hour_index", outputCol="hour_vec")
fhvhv_df = encoder_hour.fit(fhvhv_df).transform(fhvhv_df)


In [ ]:
from pyspark.sql.functions import expr

def apply_iqr_rule(df, column):
    quantiles = df.approxQuantile(column, [0.25, 0.75], 0.01)
    Q1, Q3 = quantiles
    IQR = Q3 - Q1
    threshold = (sqrt(log(df.count())) - 0.5) * IQR
    lower_bound = Q1 - threshold
    upper_bound = Q3 + threshold
    return df.filter((col(column) >= lower_bound) & (col(column) <= upper_bound))

# Apply the rule to each relevant column
fhvhv_df = apply_iqr_rule(fhvhv_df, "trip_miles")
fhvhv_df = apply_iqr_rule(fhvhv_df, "trip_time")
fhvhv_df = apply_iqr_rule(fhvhv_df, "tips")
fhvhv_df = apply_iqr_rule(fhvhv_df, "driver_pay")


In [31]:
# Example: Inspect `trip_time` min and max
fhvhv_df = fhvhv_df.filter((F.col('PULocationID') >= 1) & (F.col('PULocationID')<= 263))
get_dataset_shape(fhvhv_df)

fhvhv_df.selectExpr("min(PULocationID)", "max(PULocationID)").show()

fhvhv_df.selectExpr("min(trip_miles)", "max(trip_miles)").show()

fhvhv_df.selectExpr("min(trip_time)", "max(trip_time)").show()

fhvhv_df.selectExpr("min(trip_time)", "max(base_passenger_fare)").show()

fhvhv_df.selectExpr("min(trip_time)", "max(tolls)").show()

Dataset shape is: 116700743 rows, 24 columns


+-----------------+-----------------+
|min(PULocationID)|max(PULocationID)|
+-----------------+-----------------+
|                1|              263|
+-----------------+-----------------+



+---------------+---------------+
|min(trip_miles)|max(trip_miles)|
+---------------+---------------+
|            0.0|         568.43|
+---------------+---------------+



+--------------+--------------+
|min(trip_time)|max(trip_time)|
+--------------+--------------+
|             0|         94853|
+--------------+--------------+



In [40]:
from pyspark.sql.functions import col, expr
from pyspark.sql.functions import round

# Avoid division by zero by filtering out trips with zero duration

# Calculate earnings per hour (Response Variable)
# 2 decimals places
fhvhv_df = fhvhv_df.withColumn("earnings_per_hour", round(((col("driver_pay") + col("tips")) / (col("trip_time") / 3600)), 2))

# Preview to check that it worked
fhvhv_df.limit(5)


hvfhs_license_num,pickup_datetime,PULocationID,trip_miles,trip_time,tips,driver_pay,earnings_per_hour
HV0005,2023-05-01 00:09:39,32,2.247,685,0.0,10.27,53.97
HV0005,2023-05-01 00:43:35,81,1.826,493,0.0,7.75,56.59
HV0003,2023-05-01 00:06:54,121,3.88,810,1.0,12.71,60.93
HV0003,2023-05-01 00:24:27,92,2.31,694,0.0,10.57,54.83
HV0003,2023-05-01 00:39:13,53,9.45,1074,0.0,24.27,81.35


In [ ]:

# Convert the date column to a proper datetime format (assuming the column is named 'date')
combined_holidays_df = holidays_df.withColumn('date', to_date(holidays_df['date'], 'yyyy-MM-dd'))

In [ ]:
fhvhv_df.cache()